# Carrinho IESB — agente com busca, aprendizado e LLM (Fases 1, 2 e 3)

As três fases no mesmo notebook, seguindo o gabarito da Aula 10: o A* encontra a rota (Fase 1), um modelo treinado com dados prevê o tempo de cada trecho (Fase 2) e o Qwen entende o pedido, chama as ferramentas e explica a rota em português (Fase 3). Execute de cima para baixo.

**Estimativas didáticas:** velocidade base fixa de 15 km/h, distâncias aproximadas e atrasos sintéticos. A velocidade não é uma entrada da consulta nem uma medição física. O tempo previsto inclui atrasos e não é apenas distância/velocidade.

O grafo tem 90 nós e 90 ligações, incluindo um ciclo no pátio. No Marco 1, comparamos buscas por distância; no agente final, o A* minimiza o tempo previsto em segundos e informa a distância da rota escolhida em metros. O programa não detecta cadeiras nem garante passagem física.


## 1. Origem e destino
Altere apenas `ORIGEM` e `DESTINO`. A constante `VELOCIDADE_BASE_KMH = 15.0` define a hipótese fixa da simulação e da geração de dados; não é parâmetro da pergunta nem da ferramenta de rota.


In [ ]:
ORIGEM = "Entrada Principal"
DESTINO = "Mesa 4"
VELOCIDADE_BASE_KMH = 15.0  # Hipótese fixa da simulação; não é entrada da consulta.


## 1.1 Modelagem do problema em espaço de estados

- **Estado:** o local atual do carrinho, representado pelo nome de um nó de `LOCAIS`. Não são guardadas coordenadas nem orientação do carrinho.
- **Estado inicial:** o local informado em `ORIGEM`.
- **Ações:** mover-se do local atual para qualquer local ligado diretamente a ele em `VIZINHOS`.
- **Função de transição:** ao escolher um vizinho, o novo estado passa a ser esse vizinho. Como as ligações são bidirecionais, o movimento pode ocorrer nos dois sentidos cadastrados.
- **Teste de objetivo:** verificar se o local atual é igual a `DESTINO`.
- **Custo de uma ação:** o comprimento aproximado, em metros, da ligação percorrida.
- **Custo do caminho:** a soma dos comprimentos aproximados de todas as ligações da origem até o destino. No Marco 1, o melhor caminho é o de menor distância estimada. Na Fase 2 e no agente final, cada ação custa o tempo previsto em segundos e o caminho minimiza a soma desses tempos; a distância é informada separadamente, sem somar metros e segundos.

O custo uniforme ordena a fronteira somente pelo custo já percorrido `g`. O A* ordena por `f = g + h`, acrescentando uma estimativa admissível do custo restante. O LLM não participa desse cálculo: ele apenas interpreta o pedido e escolhe a ferramenta; a busca calcula a rota.

## 2. Mapa e distâncias aproximadas em metros
Referência: planta enviada em 12/11/2026. As medidas são estimativas, não levantamento topográfico.

- Eixo principal: 210 m divididos em 60 + 48 + 39 + 39 + 24 m, usando aproximadamente a posição vertical dos acessos no desenho.
- Pátio: quatro pontos de circulação A/B/C/D, sendo C o nó Pátio/Direito. Trechos estimados AB=20 m, BC=18 m, CD=24 m, DA=18 m; ramais de 2 m até mesas e Easy Smart, 4 m até Café Mania. Acesso anterior ao eixo mantido em 27,5 m. Posições didáticas, sem escala; os apoios ainda usam as estimativas anteriores.
- Design, C e B: três intervalos de corredor estimados em 10 m, com ramais de 2 m para as salas; grupos 03/06, 02/05 e 01/04 a partir do eixo. Banheiros mantêm 4/7 m junto ao acesso. Portas próximas são agrupadas, sem exigir alinhamento perfeito.
- I/J: pontos de corredor nas posições estimadas -35, -25, -15, -5, 0 (acesso lateral), 10, 20, 30 e 35 m; ramais de 2 m para salas e banheiros. Acesso lateral mantido em 12 m. O comprimento entre os extremos é 70 m dentro da referência geral aproximada de 75 m.
- Cinema/Biblioteca e apoios: estimativas de circulação compatíveis com a disposição do desenho, sem medir portas individualmente. A biblioteca recebe 60 m, incluindo deslocamento ao norte e passagem até sua entrada.

O croqui tem proporções inconsistentes entre áreas; não se aplica uma única escala de pixels a todo o mapa. Os nós que têm apenas o nome do bloco são pontos de referência 1 m após o corredor. Em cada tupla abaixo, o terceiro número é o comprimento estimado do trecho. As conexões continuam bidirecionais. O rótulo D-02 do Bloco C continua cadastrado provisoriamente como C-02.

In [ ]:
LOCAIS = [
    "Entrada Principal", "Eixo - Norte", "Eixo - Design", "Eixo - Bloco C",
    "Eixo - Bloco B", "Eixo - Fundo",
    "Pátio/Direito", "Mesa 1", "Mesa 2", "Mesa 3", "Mesa 4",
    "Sala dos Professores", "Café Mania", "Easy Smart",
    "WC Acessível Pátio", "WC Feminino Pátio",
    "Cinema", "WC Cinema", "Anexo Creche", "Biblioteca",
    "Design", "Corredor Design", "D-01", "D-02", "D-03", "D-04", "D-05", "D-06",
    "WC Masculino Design", "WC Feminino Design",
    "Bloco C", "Corredor Bloco C", "C-01", "C-02", "C-03", "C-04", "C-05", "C-06",
    "WC Masculino C", "WC Feminino C",
    "Bloco B", "Corredor Bloco B", "B-01", "B-02", "B-03", "B-04", "B-05", "B-06",
    "WC Masculino B", "WC Feminino B",
    "Acesso Lateral I/J", "Corredor I/J", "Bloco I", "Bloco J",
    "I-01", "I-02", "I-03", "I-04", "I-05", "I-06",
    "J-01", "J-02", "J-03", "J-04", "J-05", "J-06",
    "WC Masculino I", "WC Feminino I", "WC Masculino J", "WC Feminino J",
]


# origem, destino, distância estimada em metros
LIGACOES = [
    ("Entrada Principal", "Eixo - Norte", 60),
    ("Eixo - Norte", "Eixo - Design", 48),
    ("Eixo - Design", "Eixo - Bloco C", 39),
    ("Eixo - Bloco C", "Eixo - Bloco B", 39),
    ("Eixo - Bloco B", "Eixo - Fundo", 24),
    ("Eixo - Norte", "Pátio/Direito", 27.5),
    ("Eixo - Norte", "Cinema", 15),
    ("Eixo - Norte", "Biblioteca", 60),
    ("Eixo - Design", "Corredor Design", 5),
    ("Eixo - Bloco C", "Corredor Bloco C", 5),
    ("Eixo - Bloco B", "Corredor Bloco B", 5),
    ("Eixo - Bloco C", "Acesso Lateral I/J", 5),
    ("Acesso Lateral I/J", "Corredor I/J", 12),
    ("Corredor Design", "Design", 1),
    ("Corredor Bloco C", "Bloco C", 1),
    ("Corredor Bloco B", "Bloco B", 1),
    ("Corredor I/J", "Bloco I", 1),
    ("Corredor I/J", "Bloco J", 1),
]

# Pátio aberto: circulação condicionada ao espaço disponível entre móveis.
# Distâncias e posições das mesas são hipóteses didáticas, não medidas.
LOCAIS.extend(["Pátio - A", "Pátio - B", "Pátio - D"])
LIGACOES.extend([
    ("Pátio - A", "Pátio - B", 20),
    ("Pátio - B", "Pátio/Direito", 18),
    ("Pátio/Direito", "Pátio - D", 24),
    ("Pátio - D", "Pátio - A", 18),
    ("Pátio - A", "Mesa 1", 2),
    ("Pátio - B", "Mesa 2", 2),
    ("Pátio - D", "Mesa 3", 2),
    ("Pátio/Direito", "Mesa 4", 2),
    ("Pátio - B", "Easy Smart", 2),
    ("Pátio - B", "Café Mania", 4),
    # Apoios ainda agregados ao ponto antigo: posições não revisadas.
    ("Pátio/Direito", "Sala dos Professores", 15),
    ("Pátio/Direito", "WC Acessível Pátio", 40),
    ("Pátio/Direito", "WC Feminino Pátio", 35),
])

for local, metros in [("WC Cinema", 20), ("Anexo Creche", 25)]:
    LIGACOES.append(("Cinema", local, metros))

# Corredores contínuos confirmados. Comprimentos abaixo são estimativas didáticas.
# Cada grupo representa portas próximas, sem afirmar alinhamento físico perfeito.
for corredor, prefixo, banheiro in [
    ("Corredor Design", "D", "Design"),
    ("Corredor Bloco C", "C", "C"),
    ("Corredor Bloco B", "B", "B"),
]:
    anterior = corredor
    for salas in [("03", "06"), ("02", "05"), ("01", "04")]:
        ponto = corredor + " - " + "/".join(salas)
        LOCAIS.append(ponto)
        LIGACOES.append((anterior, ponto, 10))
        for sala in salas:
            LIGACOES.append((ponto, prefixo + "-" + sala, 2))
        anterior = ponto
    LIGACOES.append((corredor, "WC Masculino " + banheiro, 4))
    LIGACOES.append((corredor, "WC Feminino " + banheiro, 7))

# Posição longitudinal estimada em metros a partir do acesso lateral.
PONTOS_IJ = [
    ("Corredor I/J - I-01/04", -35, ["I-01", "I-04"]),
    ("Corredor I/J - I-02/05", -25, ["I-02", "I-05"]),
    ("Corredor I/J - I-03/06", -15, ["I-03", "I-06"]),
    ("Corredor I/J - WC I", -5, ["WC Masculino I", "WC Feminino I"]),
    ("Corredor I/J", 0, []),
    ("Corredor I/J - J-01/04", 10, ["J-01", "J-04"]),
    ("Corredor I/J - J-02/05", 20, ["J-02", "J-05"]),
    ("Corredor I/J - J-03/06", 30, ["J-03", "J-06"]),
    ("Corredor I/J - WC J", 35, ["WC Masculino J", "WC Feminino J"]),
]
for indice, (ponto, posicao, destinos) in enumerate(PONTOS_IJ):
    if ponto not in LOCAIS:
        LOCAIS.append(ponto)
    if indice > 0:
        anterior, posicao_anterior, _ = PONTOS_IJ[indice - 1]
        LIGACOES.append((anterior, ponto, posicao - posicao_anterior))
    for destino in destinos:
        LIGACOES.append((ponto, destino, 2))

VIZINHOS = {}
for local in LOCAIS:
    VIZINHOS[local] = []
for origem, destino, metros in LIGACOES:
    VIZINHOS[origem].append((destino, metros))
    VIZINHOS[destino].append((origem, metros))

print(len(LOCAIS), "nós e", len(LIGACOES), "ligações com distância em metros.")

## 3. A*, custo uniforme e cálculo do tempo
O laço de busca mantém o estilo do gabarito: lista `fronteira`, seleção com `min`, lista de visitados e soma dos custos. A função `buscar` permite ligar a heurística (A*) ou desligá-la (custo uniforme), além de contar nós expandidos. Um nó é contado quando seus vizinhos são examinados; o destino não é expandido.

**Heurística do IESB:** uma tabela guarda a menor distância de cada local até a Entrada Principal, calculada uma vez com custo uniforme. Usamos `h(local, destino) = abs(distância_entrada[local] - distância_entrada[destino])`. Pela desigualdade triangular, essa diferença nunca supera a menor distância entre local e destino. Também é consistente: em cada ligação (u,v) de custo c, `h(u) ≤ c + h(v)`. A tabela se baseia nos mesmos metros estimados do grafo. Reexecute esta célula se mudar as distâncias.

A preparação tem custo e seu tempo é mostrado separadamente; os gráficos medem as consultas após essa preparação. A tabela completa permanece admissível nos submapas porque remover nós não encurta caminhos. Não são coordenadas geográficas nem medidas em linha reta.

Conversão do tempo de deslocamento: `tempo (s) = distância (m) × 3,6 ÷ velocidade (km/h)`. Como a velocidade é constante, minimizar metros também minimiza esse tempo.

In [ ]:
from time import perf_counter

# A tabela é preparada antes das consultas A*.
DISTANCIA_ENTRADA = {}


def h(local, destino):
    if local not in DISTANCIA_ENTRADA or destino not in DISTANCIA_ENTRADA:
        return 0
    return abs(DISTANCIA_ENTRADA[local] - DISTANCIA_ENTRADA[destino])


def buscar(origem, destino, grafo, usar_heuristica, referencia=None):
    if referencia is None:
        referencia = DISTANCIA_ENTRADA
    if origem not in grafo or destino not in grafo:
        return None, None, 0
    estimativa = abs(referencia.get(origem, 0) - referencia.get(destino, 0)) if usar_heuristica else 0
    fronteira = [(estimativa, 0, origem, [origem])]
    visitados = []
    expandidos = 0
    while len(fronteira) > 0:
        melhor = min(fronteira)
        fronteira.remove(melhor)
        f, g, local, caminho = melhor
        if local == destino:
            return caminho, g, expandidos
        if local in visitados:
            continue
        visitados.append(local)
        expandidos = expandidos + 1
        for vizinho, custo in grafo[local]:
            if vizinho not in visitados:
                novo_g = g + custo
                estimativa = abs(referencia.get(vizinho, 0) - referencia.get(destino, 0)) if usar_heuristica else 0
                fronteira.append((novo_g + estimativa, novo_g,
                                  vizinho, caminho + [vizinho]))
    return None, None, expandidos


def a_estrela(origem, destino):
    caminho, custo, expandidos = buscar(origem, destino, VIZINHOS, True)
    return caminho, custo


# Custo uniforme usa o mesmo laço com h = 0. Não depende desta tabela.
# Pré-processamento: calcula distâncias até um ponto de referência fixo.
inicio_preparo = perf_counter()
for local in LOCAIS:
    caminho, metros, expandidos = buscar("Entrada Principal", local, VIZINHOS, False)
    DISTANCIA_ENTRADA[local] = metros
TEMPO_PREPARO_MS = (perf_counter() - inicio_preparo) * 1000
print(f"Preparação da tabela de referência: {TEMPO_PREPARO_MS:.3f} ms (uma vez).")


def calcular_tempo(distancia_m, velocidade_kmh):
    if not isinstance(velocidade_kmh, (int, float)) or isinstance(velocidade_kmh, bool):
        raise ValueError("A velocidade precisa ser um número maior que zero.")
    if not 0 < velocidade_kmh < float("inf"):
        raise ValueError("A velocidade precisa ser finita e maior que zero.")
    if not 0 <= distancia_m < float("inf"):
        raise ValueError("A distância precisa ser finita e não negativa.")
    return distancia_m * 3.6 / velocidade_kmh

## 4. Referência didática por distância e consulta de vizinhos
`calcular_rota` é uma referência do Marco 1: menor distância e tempo ideal sem atrasos a 15 km/h. Não é exposta ao Qwen. A única ferramenta de rota do agente será `rota_tempo_previsto`, definida após o treinamento na seção 5.3. `listar_vizinhos` consulta ligações diretas.


In [ ]:
def calcular_rota(origem: str, destino: str) -> str:
    """
    Calcula com A* a menor distância estimada no grafo e o tempo a velocidade constante.

    Args:
        origem: nome exato do local de partida na lista de locais
        destino: nome exato do local de chegada na lista de locais
    """
    if origem not in LOCAIS or destino not in LOCAIS:
        return "Erro: origem ou destino não existe no mapa. Consulte a lista LOCAIS."
    try:
        calcular_tempo(0, VELOCIDADE_BASE_KMH)
    except ValueError as erro:
        return "Erro: " + str(erro)
    caminho, distancia = a_estrela(origem, destino)
    if caminho is None:
        return "Não existe caminho entre " + origem + " e " + destino + "."
    segundos = calcular_tempo(distancia, VELOCIDADE_BASE_KMH)
    return ("Caminho: " + " -> ".join(caminho)
            + f". Distância aproximada: {distancia:.1f} m."
            + f" Velocidade constante: {VELOCIDADE_BASE_KMH:g} km/h."
            + f" Tempo estimado: {segundos:.1f} s ({segundos / 60:.2f} min), sem paradas.")


def listar_vizinhos(local: str) -> str:
    """
    Lista os locais ligados diretamente a um local da planta do IESB.

    Args:
        local: nome escrito pelo usuário; a aplicação normaliza e pede esclarecimento se ambíguo
    """
    if local not in LOCAIS:
        return "Erro: o local " + str(local) + " não existe no mapa."
    partes = []
    for vizinho, custo in VIZINHOS[local]:
        partes.append(vizinho)
    return "Vizinhos de " + local + ": " + ", ".join(partes) + "."





## 5. Referência do Marco 1: distância e tempo ideal a 15 km/h
Esta parte funciona sem o Qwen. O cálculo ideal não inclui atrasos e serve apenas como referência didática. A ferramenta final do agente usa o tempo aprendido da seção 5.3.


In [ ]:
print(calcular_rota(ORIGEM, DESTINO))
for origem, destino in [("Entrada Principal", "Mesa 1"),
                        ("Biblioteca", "C-02"), ("B-04", "I-05")]:
    print(calcular_rota(origem, destino))


## 5.1 Experimento do Marco 1 — tabela e gráficos
Comparamos custo uniforme e A* em três **recortes conectados da mesma planta**, com 20, 40 e 70 nós. Não são três plantas independentes. Em cada recorte, ambas as buscas usam a entrada como origem e o local mais distante dela como destino; o destino pode mudar entre recortes. Esta é uma amostra limitada, de uma consulta por tamanho.

O tempo de execução é medido no computador com `perf_counter`, em milissegundos: mediana de 30 consultas após aquecimento, sem incluir preparação da heurística nem desenho do gráfico. É diferente do **tempo de deslocamento do carrinho**, estimado em segundos a partir da velocidade. Os tempos de execução variam entre máquinas e execuções; menos expansões não garante menor tempo.

As buscas devem retornar o mesmo custo ótimo em metros; caminhos empatados podem diferir. O grafo contém um ciclo no pátio. A referência na entrada pode gerar muitos empates de prioridade. Consulte as contagens da execução abaixo; os resultados anteriores à subdivisão dos corredores não representam esta versão. Não se presume vantagem de tempo.

**Análise dos resultados e limites:**

- O custo uniforme e o A* encontraram o mesmo custo ótimo nos três recortes, como esperado para implementações corretas sobre o mesmo grafo.
- A heurística é admissível e consistente, mas isso garante a qualidade da solução, não uma redução obrigatória de expansões ou de tempo.
- A heurística usa uma única referência, a Entrada Principal. Em uma árvore, vários nós podem receber o mesmo valor de `f`, criando empates; a ordem de desempate da tupla também influencia quais nós são examinados primeiro.
- A amostra contém somente uma consulta por tamanho. Ela não permite generalizar que os dois algoritmos sempre expandem a mesma quantidade de nós nem que um deles é sempre mais rápido.
- Os tamanhos 20, 40 e 70 são recortes conectados da mesma planta, e não três mapas independentes. Isso mantém a comparação controlada, mas atende ao texto do enunciado de forma mais limitada do que três topologias diferentes.
- A mediana de 30 repetições reduz a influência de picos ocasionais, porém os tempos são muito pequenos e dependem da máquina, do ambiente e da carga do Colab.
- O tempo de preparação de `DISTANCIA_ENTRADA` é informado separadamente. Ignorá-lo faria a comparação parecer mais favorável ao A* quando há poucas consultas; ele pode ser diluído quando a mesma tabela é reutilizada muitas vezes.
- O tempo estimado do carrinho a 10 ou 15 km/h não é o tempo de execução da busca e não foi medido em campo.

In [ ]:
from statistics import median


def recortar_mapa(quantidade):
    # Seleciona nós conectados a partir da entrada, sem inventar ligações.
    selecionados = ["Entrada Principal"]
    fila = ["Entrada Principal"]
    while fila and len(selecionados) < quantidade:
        local = fila.pop(0)
        for vizinho, metros in VIZINHOS[local]:
            if vizinho not in selecionados and len(selecionados) < quantidade:
                selecionados.append(vizinho)
                fila.append(vizinho)
    grafo = {}
    for local in selecionados:
        grafo[local] = []
        for vizinho, metros in VIZINHOS[local]:
            if vizinho in selecionados:
                grafo[local].append((vizinho, metros))
    return grafo


RESULTADOS = []
REPETICOES = 30
for tamanho in [20, 40, 70]:
    grafo = recortar_mapa(tamanho)
    origem = "Entrada Principal"
    destino = max(grafo, key=lambda local: DISTANCIA_ENTRADA[local])
    print(f"Mapa de {len(grafo)} nós: {origem} -> {destino}")
    for algoritmo, usar_h in [("Custo uniforme", False), ("A*", True)]:
        buscar(origem, destino, grafo, usar_h)  # aquecimento fora da medição
        tempos = []
        for repeticao in range(REPETICOES):
            inicio = perf_counter()
            caminho, custo, expandidos = buscar(origem, destino, grafo, usar_h)
            tempos.append((perf_counter() - inicio) * 1000)
        RESULTADOS.append({"nos": len(grafo), "algoritmo": algoritmo,
                           "origem": origem, "destino": destino,
                           "expandidos": expandidos, "custo_m": custo,
                           "execucao_ms": median(tempos)})

print("\nNós | Algoritmo      | Expandidos | Custo (m) | Execução mediana (ms)")
for r in RESULTADOS:
    print(f"{r['nos']:>3} | {r['algoritmo']:<14} | {r['expandidos']:>10} | "
          f"{r['custo_m']:>9.1f} | {r['execucao_ms']:.4f}")
print(f"Pré-processamento da heurística, fora das consultas: {TEMPO_PREPARO_MS:.3f} ms.")

## 5.2 Gráficos
As figuras aparecem abaixo e são salvas como PNG na pasta de execução do notebook. Matplotlib costuma estar disponível no Colab; a célula seguinte instala a biblioteca quando necessário. Os três primeiros gráficos usam as medições da tabela; o quarto compara o tempo estimado de deslocamento a 10 e 15 km/h.

In [ ]:
%pip -q install matplotlib

In [ ]:
import matplotlib.pyplot as plt

for campo, titulo, unidade in [
    ("expandidos", "Nós expandidos por busca", "Nós expandidos"),
    ("custo_m", "Custo da rota encontrada", "Distância estimada (m)"),
    ("execucao_ms", "Tempo de execução da busca (mediana de 30 consultas)", "Tempo de execução observado (ms)"),
]:
    plt.figure(figsize=(8, 4))
    for algoritmo, deslocamento, cor in [("Custo uniforme", -0.2, "#4777b3"),
                                        ("A*", 0.2, "#dc8934")]:
        valores = [r[campo] for r in RESULTADOS if r["algoritmo"] == algoritmo]
        barras = plt.bar([x + deslocamento for x in range(3)], valores,
                        width=0.4, label=algoritmo, color=cor)
        for barra, valor in zip(barras, valores):
            texto = f"{valor:.3f}" if campo == "execucao_ms" else f"{valor:g}"
            plt.text(barra.get_x() + barra.get_width() / 2, valor, texto,
                     ha="center", va="bottom", fontsize=9)
    plt.xticks([0, 1, 2], ["20 nós", "40 nós", "70 nós"])
    plt.ylabel(unidade)
    plt.title(titulo)
    plt.margins(y=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig("comparacao_" + campo + ".png", dpi=150)
    plt.show()
    plt.close()

caminho, distancia = a_estrela(ORIGEM, DESTINO)
if caminho is not None:
    velocidades = [VELOCIDADE_BASE_KMH]
    tempos = [calcular_tempo(distancia, velocidade) for velocidade in velocidades]
    plt.figure(figsize=(8, 4))
    plt.bar(["Base fixa: 15 km/h"], tempos, color="#4777b3")
    for x, segundos in enumerate(tempos):
        plt.text(x, segundos, f"{segundos:.1f} s", ha="center", va="bottom")
    plt.ylabel("Tempo ideal sem atrasos (s) — referência do Marco 1")
    plt.title(f"{ORIGEM} → {DESTINO}\nDistância aproximada: {distancia:.1f} m; velocidade constante, sem paradas")
    plt.margins(y=0.2)
    plt.tight_layout()
    plt.savefig("tempo_deslocamento.png", dpi=150)
    plt.show()
    plt.close()

## 5.3 Fase 2 — aprendizado com dados sintéticos

Escolha autorizada pelo grupo: dados sintéticos. São 90 trechos × 2 cenários × 12 repetições = 2.160 linhas. Piso e porta são atributos fictícios sorteados por trecho; fluxo e obstáculo variam por observação. Nenhuma dessas características foi medida no campus. Não há simulação de movimento do carrinho.

Regra pública do alvo (s): distância × 3,6 / 15 × (1 + 0,15 × fluxo + 0,25 × piso irregular) + 3 × porta + 12 × obstáculo + ruído normal de desvio 0,8 s; mínimo 0,1 s. Semente 42. Obstáculo significa atraso no cenário fictício, não bloqueio inacessível. Tempos e previsões são SINTÉTICOS.

Não há faltantes ou linhas descartadas nesta geração. Imputação por mediana/moda, padronização e one-hot são ajustados dentro de cada divisão de treino via Pipeline. Nomes e identificador de trecho não entram no modelo. Reservamos 20% dos trechos para teste; comparamos regressão linear e árvore em validação cruzada de 5 grupos nos demais trechos. Selecionamos pelo MAE, avaliamos uma vez no teste reservado e só então reajustamos o modelo escolhido com todos os dados para uso da ferramenta.

A comparação mede MAE/RMSE em segundos e R². Os erros e o CSV são salvos em resultados_fase2_15kmh. A função rota_tempo_previsto usa o mesmo laço A*, com pesos e heurística em segundos. Os dois sentidos têm o mesmo tempo por hipótese. A consulta usa fluxo 1/4 e obstáculo 0/1 para tranquilo/movimentado. A velocidade base fixa de 15 km/h faz parte da regra de geração; esta ferramenta não recebe velocidade. O custo é somente tempo previsto em segundos. A distância informada é a soma dos metros da rota escolhida por tempo, não necessariamente a menor distância do grafo.

O grafo de 90 nós possui um ciclo no pátio. A Fase 2 pode escolher outro caminho pelos custos previstos. As passagens dependem do espaço disponível entre mesas e cadeiras. Os testes locais não substituem a execução integral no Colab nem avaliam validade física.

In [ ]:
%pip -q install numpy pandas scikit-learn

In [ ]:
# FASE_2_SINTETICA: executada também dentro do notebook unificado.
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMENTE_DADOS = 42
PASTA_FASE2 = Path('resultados_fase2_15kmh')
PASTA_FASE2.mkdir(exist_ok=True)


def gerar_dados_sinteticos():
    sorteio = random.Random(SEMENTE_DADOS)
    linhas = []
    for trecho, (origem, destino, distancia) in enumerate(LIGACOES):
        # Atributos fictícios: não descrevem portas/pisos observados no campus.
        piso = sorteio.choice(['regular', 'irregular'])
        porta = sorteio.choice([0, 1])
        for cenario in ['tranquilo', 'movimentado']:
            for repeticao in range(12):
                fluxo = sorteio.randint(0, 2) if cenario == 'tranquilo' else sorteio.randint(3, 5)
                obstaculo = int(sorteio.random() < (0.1 if cenario == 'tranquilo' else 0.4))
                base = distancia * 3.6 / VELOCIDADE_BASE_KMH
                tempo = (base * (1 + 0.15 * fluxo + 0.25 * (piso == 'irregular'))
                         + 3 * porta + 12 * obstaculo + sorteio.gauss(0, 0.8))
                linhas.append(dict(trecho=trecho, origem=origem, destino=destino,
                                   distancia_m=distancia, cenario=cenario, fluxo=fluxo,
                                   piso=piso, porta=porta, obstaculo=obstaculo,
                                   repeticao=repeticao, tempo_s=max(0.1, tempo),
                                   origem_do_dado='sintetico', velocidade_base_kmh=VELOCIDADE_BASE_KMH))
    return pd.DataFrame(linhas)


NUMERICAS = ['distancia_m', 'fluxo', 'porta', 'obstaculo']
CATEGORICAS = ['piso', 'cenario']
ENTRADAS = NUMERICAS + CATEGORICAS


def preparar_modelo(nome):
    numericas = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
    categorias = make_pipeline(SimpleImputer(strategy='most_frequent'),
                               OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    preparo = ColumnTransformer([('numericas', numericas, NUMERICAS),
                                 ('categorias', categorias, CATEGORICAS)])
    estimador = LinearRegression() if nome == 'regressao_linear' else DecisionTreeRegressor(
        max_depth=8, min_samples_leaf=5, random_state=SEMENTE_DADOS)
    return make_pipeline(preparo, estimador)


def metricas_tempo(real, previsto):
    return dict(mae_s=float(mean_absolute_error(real, previsto)),
                rmse_s=float(np.sqrt(mean_squared_error(real, previsto))),
                r2=float(r2_score(real, previsto)))


DADOS_TEMPO = gerar_dados_sinteticos()
# Não há faltantes ou descartes na geração. A imputação é aprendida dentro de cada fold.
assert not DADOS_TEMPO.isna().any().any()
assert not DADOS_TEMPO.duplicated(['trecho', 'cenario', 'repeticao']).any()
assert (DADOS_TEMPO.tempo_s > 0).all()
DADOS_TEMPO.to_csv(PASTA_FASE2 / 'tempos_sinteticos.csv', index=False)

separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMENTE_DADOS)
indices_treino, indices_teste = next(separador.split(DADOS_TEMPO, groups=DADOS_TEMPO.trecho))
treino = DADOS_TEMPO.iloc[indices_treino]
teste = DADOS_TEMPO.iloc[indices_teste]
assert set(treino.trecho).isdisjoint(teste.trecho)
COMPARACAO_MODELOS = []
for nome in ['regressao_linear', 'arvore_decisao']:
    previsto = cross_val_predict(preparar_modelo(nome), treino[ENTRADAS], treino.tempo_s,
                                 groups=treino.trecho, cv=GroupKFold(n_splits=5))
    previsto = np.maximum(0.1, previsto)
    COMPARACAO_MODELOS.append(dict(modelo=nome, **metricas_tempo(treino.tempo_s, previsto)))
MELHOR_MODELO = min(COMPARACAO_MODELOS, key=lambda r: r['mae_s'])['modelo']
modelo_avaliado = preparar_modelo(MELHOR_MODELO)
modelo_avaliado.fit(treino[ENTRADAS], treino.tempo_s)
previsoes_teste = np.maximum(0.1, modelo_avaliado.predict(teste[ENTRADAS]))
METRICAS_TESTE = metricas_tempo(teste.tempo_s, previsoes_teste)
erros = teste.copy()
erros['previsto_s'] = previsoes_teste
erros['erro_absoluto_s'] = abs(erros.tempo_s - erros.previsto_s)
erros.to_csv(PASTA_FASE2 / 'previsoes_teste.csv', index=False)
pd.DataFrame(COMPARACAO_MODELOS).to_csv(PASTA_FASE2 / 'validacao_cruzada.csv', index=False)
print('Dados SINTÉTICOS:', len(DADOS_TEMPO), 'linhas;', len(LIGACOES), 'trechos; 24 cenários/repetições por trecho.')
print('Validação cruzada por trecho (5 folds), apenas no treino:')
print(pd.DataFrame(COMPARACAO_MODELOS).to_string(index=False))
print('Escolhido pelo menor MAE de validação:', MELHOR_MODELO)
print('Teste reservado por trecho, usado somente depois da escolha:', METRICAS_TESTE)
print('Erro médio absoluto por cenário no teste:')
print(erros.groupby('cenario').erro_absoluto_s.mean())
print('Cinco maiores erros no teste:')
print(erros.nlargest(5, 'erro_absoluto_s')[['origem', 'destino', 'cenario', 'tempo_s', 'previsto_s', 'erro_absoluto_s']].to_string(index=False))
(PASTA_FASE2 / 'resumo.json').write_text(json.dumps(dict(
    semente=SEMENTE_DADOS, velocidade_base_kmh=VELOCIDADE_BASE_KMH,
    custo_rota='tempo_previsto_s', linhas=len(DADOS_TEMPO), trechos_treino=sorted(treino.trecho.unique().tolist()),
    trechos_teste=sorted(teste.trecho.unique().tolist()), validacao=COMPARACAO_MODELOS,
    escolhido=MELHOR_MODELO, teste=METRICAS_TESTE), ensure_ascii=False, indent=2))

# Reajuste final para uso da ferramenta. As métricas acima são anteriores a esse reajuste.
MODELO_TEMPO = preparar_modelo(MELHOR_MODELO)
MODELO_TEMPO.fit(DADOS_TEMPO[ENTRADAS], DADOS_TEMPO.tempo_s)


def prever_grafo_tempo(cenario):
    if cenario not in ['tranquilo', 'movimentado', 'cadeiras_no_patio']:
        raise ValueError('Cenário desconhecido.')
    atributos = DADOS_TEMPO.drop_duplicates('trecho')[ENTRADAS].reset_index(drop=True).copy()
    base = 'tranquilo' if cenario == 'cadeiras_no_patio' else cenario
    atributos['cenario'] = base
    atributos['fluxo'] = 1 if base == 'tranquilo' else 4
    atributos['obstaculo'] = 0 if base == 'tranquilo' else 1
    if cenario == 'cadeiras_no_patio':
        # Cenário didático: atraso no trecho A-B, sem alegar bloqueio real detectado.
        for indice, (origem, destino, _) in enumerate(LIGACOES):
            if {origem, destino} == {'Pátio - A', 'Pátio - B'}:
                atributos.loc[indice, ['cenario', 'fluxo', 'obstaculo']] = ['movimentado', 5, 1]
    tempos = np.maximum(0.1, MODELO_TEMPO.predict(atributos))
    assert len(tempos) == len(LIGACOES)
    grafo = {local: [] for local in LOCAIS}
    for (origem, destino, distancia), segundos in zip(LIGACOES, tempos):
        grafo[origem].append((destino, float(segundos)))
        grafo[destino].append((origem, float(segundos)))
    return grafo


def rota_tempo_previsto(origem: str, destino: str, cenario: str = 'tranquilo') -> str:
    """Consulta caminho, distância e menor tempo previsto, com base fixa de 15 km/h.

    Use para todos os pedidos de rota, distância e tempo. Os atrasos são sintéticos.
    Não recebe velocidade: 15 km/h é a base fixa da simulação, não a média da viagem.

    Args:
        origem: local de partida como escrito pelo usuário; a aplicação normaliza antes de executar
        destino: local de chegada como escrito pelo usuário; a aplicação normaliza antes de executar
        cenario: cenário fictício; omitir para tranquilo (choices: ["tranquilo", "movimentado", "cadeiras_no_patio"])
    """
    if origem not in LOCAIS or destino not in LOCAIS:
        return 'Erro: origem ou destino não existe no mapa.'
    if cenario not in ['tranquilo', 'movimentado', 'cadeiras_no_patio']:
        return 'Erro: cenário deve ser tranquilo, movimentado ou cadeiras_no_patio.'
    grafo = prever_grafo_tempo(cenario)
    # Referência em SEGUNDOS: nunca reutilizar a heurística antiga em metros.
    referencia = {local: buscar('Entrada Principal', local, grafo, False)[1] for local in LOCAIS}
    caminho, segundos, expandidos = buscar(origem, destino, grafo, True, referencia)
    if caminho is None:
        return 'Não existe caminho entre os locais.'
    distancia = sum(dict(VIZINHOS[a])[b] for a, b in zip(caminho, caminho[1:]))
    aviso = 'Origem e destino coincidem; não há deslocamento no grafo.\n' if origem == destino else ''
    return (aviso + 'Caminho: ' + ' -> '.join(caminho)
            + f'\nDistância aproximada desta rota: {distancia:.1f} m.'
            + f'\nTempo previsto com atrasos: {segundos:.1f} s.'
            + f'\nVelocidade base fixa: {VELOCIDADE_BASE_KMH:g} km/h.'
            + '\nCenário SINTÉTICO: ' + cenario + '. Modelo: ' + MELHOR_MODELO
            + '.\nSem validação de tempo no campus; conferir passagem entre móveis.')


for cenario in ['tranquilo', 'movimentado']:
    print(rota_tempo_previsto(ORIGEM, DESTINO, cenario))


def demonstrar_alternativa_patio():
    origem, destino = 'Mesa 1', 'Mesa 4'
    curto, metros = a_estrela(origem, destino)
    grafo = prever_grafo_tempo('cadeiras_no_patio')
    referencia = {local: buscar('Entrada Principal', local, grafo, False)[1] for local in LOCAIS}
    rapido, tempo, _ = buscar(origem, destino, grafo, True, referencia)
    tempo_curto = sum(dict(grafo[a])[b] for a, b in zip(curto, curto[1:]))
    metros_rapido = sum(dict(VIZINHOS[a])[b] for a, b in zip(rapido, rapido[1:]))
    resultado = dict(origem=origem, destino=destino, cenario='cadeiras_no_patio',
                     velocidade_base_kmh=VELOCIDADE_BASE_KMH,
                     rota_distancia=curto, distancia_m=metros, tempo_rota_curta_s=tempo_curto,
                     rota_tempo=rapido, distancia_rota_tempo_m=metros_rapido, tempo_previsto_s=tempo,
                     mudou=curto != rapido, economia_prevista_s=tempo_curto - tempo)
    print('Demonstração SINTÉTICA; requer passagem física disponível:', resultado)
    (PASTA_FASE2 / 'demonstracao_patio.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=2))
    return resultado


DEMONSTRACAO_PATIO = demonstrar_alternativa_patio()

FERRAMENTAS = [rota_tempo_previsto, listar_vizinhos]


## 6. Qwen do gabarito
A partir daqui é necessário instalar `transformers` e baixar o modelo público. No Colab, use uma GPU T4 se disponível. As funções anteriores já calculam os resultados sem o LLM.

In [ ]:
%pip -q install transformers

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    dispositivo = "cuda"
else:
    dispositivo = "cpu"
print("Dispositivo:", dispositivo)

nome_qwen = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizador_qwen = AutoTokenizer.from_pretrained(nome_qwen)
if dispositivo == "cuda":
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float16).to(dispositivo)
else:
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float32)
qwen.eval()


ULTIMA_GERACAO = {}


def conversar(mensagens, ferramentas=None, max_tokens=200):
    global ULTIMA_GERACAO
    entrada = tokenizador_qwen.apply_chat_template(
        mensagens, tools=ferramentas, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(dispositivo)
    with torch.no_grad():
        saida = qwen.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    tamanho_da_pergunta = entrada["input_ids"].shape[-1]
    novos = saida[0][tamanho_da_pergunta:]
    eos = qwen.generation_config.eos_token_id
    eos = eos if isinstance(eos, list) else [eos]
    terminou_eos = bool(len(novos)) and int(novos[-1]) in eos
    motivo = 'eos' if terminou_eos else ('limite_tokens' if len(novos) >= max_tokens else 'outro')
    ULTIMA_GERACAO = dict(tokens_gerados=len(novos), limite_tokens=max_tokens, motivo_termino=motivo)
    return tokenizador_qwen.decode(novos, skip_special_tokens=True)


print("Qwen carregado.")

## 7. O ciclo do agente: pensar → agir → observar → responder
O agente segue o mesmo ciclo do gabarito da Aula 10, com limite de 3 passos:

1. **Pensar:** o Qwen lê a pergunta e decide se precisa de uma ferramenta. Se precisar, escreve o pedido entre `<tool_call>` e `</tool_call>`.
2. **Agir:** o código lê o pedido, confere os nomes e executa a ferramenta. O Qwen nunca calcula a rota: quem calcula é o A* com os tempos aprendidos na Fase 2.
3. **Observar:** o resultado volta para o Qwen como mensagem com o papel `"tool"`.
4. **Responder:** o Qwen explica a rota em português, copiando caminho, metros e segundos do resultado.

**O que o código confere antes de executar:** `padronizar` tira acentos, maiúsculas e pontuação, transforma `b4` em `b 4` e troca palavras como `banheiro` → `wc`. Depois `resolver_local` procura o nome do mapa, um apelido ou, para erro de digitação sem número, um nome parecido (`get_close_matches`). Se o local for genérico (“banheiro”), ausente ou desconhecido, o programa pergunta ao usuário em vez de escolher. Pedido com JSON inválido ou nome que não aparece na pergunta volta para o Qwen como erro, e ele pode corrigir.

**Resposta sem ferramenta:** o Qwen 1.5B às vezes responde uma pergunta sobre o mapa sem chamar a ferramenta e inventa rota e números. Por isso, se a pergunta cita um local ou uma palavra de rota (`fala_do_mapa`) e o Qwen respondeu sem ferramenta, o código pede de novo. Se ele não chamar a ferramenta em 3 passos, o agente diz que não conseguiu, em vez de mostrar uma rota inventada.

**Controle de alucinação:** depois da explicação, `numeros_fora_da_ferramenta` procura números que não aparecem no retorno da ferramenta (por exemplo, converter 22,3 s em “3 minutos”). Se encontrar, o agente avisa e mostra o resultado exato. É uma triagem simples, não uma garantia de que o Qwen entendeu a intenção do usuário.

`retornar_detalhes=True` devolve também as chamadas executadas, as respostas brutas do Qwen e os números sinalizados.


In [ ]:
# Agente: o Qwen pede a ferramenta, o código confere e executa, e o Qwen explica o resultado.
import json
import re
import unicodedata
from difflib import get_close_matches

# Palavras trocadas ou ignoradas ao comparar nomes ('' = ignorar).
TROCAS = {'banheiro': 'wc', 'fem': 'feminino', 'masc': 'masculino',
          'sala': '', 'bloco': '', 'do': '', 'da': '', 'de': '', 'dos': '', 'das': ''}


def padronizar(texto):
    # "Sala B04" -> "b 4"; "banheiro fem do Design" -> "wc feminino design"
    texto = unicodedata.normalize('NFD', str(texto).lower())
    texto = ''.join(letra for letra in texto if unicodedata.category(letra) != 'Mn')  # tira acentos
    texto = re.sub(r'[^a-z0-9]+', ' ', texto)           # tira pontuação
    texto = re.sub(r'([a-z])([0-9])', r'\1 \2', texto)  # b4 -> b 4
    palavras = []
    for palavra in texto.split():
        palavra = TROCAS.get(palavra, palavra)
        if palavra.isdigit():
            palavra = str(int(palavra))                  # 04 -> 4
        if palavra:
            palavras.append(palavra)
    return ' '.join(palavras)


# Apelidos levam a um único local. Não criam nós nem ligações no mapa.
APELIDOS = {'biblio': 'Biblioteca', 'bibl': 'Biblioteca', 'entrada': 'Entrada Principal',
            'patio': 'Pátio/Direito', 'easy': 'Easy Smart',
            'mesa um': 'Mesa 1', 'mesa dois': 'Mesa 2', 'mesa tres': 'Mesa 3', 'mesa quatro': 'Mesa 4'}

# Nomes genéricos: o programa pergunta qual local o usuário quer.
GENERICOS = {'banheiro': [n for n in LOCAIS if n.startswith('WC ')],
             'lanchonete': ['Easy Smart', 'Café Mania'],
             'cantina': ['Easy Smart', 'Café Mania'],
             'cafe': ['Easy Smart', 'Café Mania']}

NOMES = {}
for local in LOCAIS:
    NOMES[padronizar(local)] = local
assert len(NOMES) == len(LOCAIS), 'Dois locais ficaram com o mesmo nome padronizado.'
for apelido, local in APELIDOS.items():
    NOMES[padronizar(apelido)] = local
GENERICOS = {padronizar(nome): opcoes for nome, opcoes in GENERICOS.items()}

CENARIOS = {'tranquilo': ['tranquilo', 'calmo', 'normal', 'padrao'],
            'movimentado': ['movimentado', 'cheio', 'lotado', 'fluxo intenso'],
            'cadeiras_no_patio': ['cadeiras no patio', 'cadeiras']}


def resolver_local(texto):
    # Devolve (local, opções). local None = precisa perguntar ao usuário.
    chave = padronizar(texto)
    if chave in NOMES:
        return NOMES[chave], []
    if chave in GENERICOS:
        return None, GENERICOS[chave]
    if chave == '' or re.search(r'[0-9]', chave):
        return None, []                    # número de sala ou mesa nunca é "corrigido"
    parecidos = get_close_matches(chave, list(NOMES), n=2, cutoff=0.85)   # erro de digitação
    if len(parecidos) == 1:
        return NOMES[parecidos[0]], []
    return None, [NOMES[p] for p in parecidos]


def resolver_cenario(texto):
    variantes = {}
    for cenario, palavras in CENARIOS.items():
        for palavra in [cenario] + palavras:
            variantes[padronizar(palavra)] = cenario
    parecidos = get_close_matches(padronizar(texto), list(variantes), n=1, cutoff=0.7)  # aceita "tranqulho" e "tranq"
    return variantes[parecidos[0]] if parecidos else None


def aparece_na_pergunta(textos, pergunta):
    pergunta = ' ' + padronizar(pergunta) + ' '
    return any(padronizar(t) and ' ' + padronizar(t) + ' ' in pergunta for t in textos)


def local_na_pergunta(local, pergunta):
    # O local precisa ter sido escrito pelo usuário, mesmo abreviado ou com erro de digitação:
    # testa cada trecho de 1 a 4 palavras da pergunta com o mesmo resolver_local.
    palavras = padronizar(pergunta).split()
    for tamanho in range(1, 5):
        for inicio in range(len(palavras) - tamanho + 1):
            trecho = ' '.join(palavras[inicio:inicio + tamanho])
            if resolver_local(trecho)[0] == local:
                return True
    return False


def pergunta_ao_usuario(campo, opcoes):
    mensagem = f'Qual é o local de {campo}?' if campo != 'local' else 'Qual local deseja consultar?'
    if opcoes:
        return mensagem + ' Opções: ' + ', '.join(opcoes) + '.'
    return mensagem + ' Use um nome do mapa; para sala, informe bloco e número (ex.: B-04).'


CAMPOS = {'rota_tempo_previsto': ['origem', 'destino'], 'listar_vizinhos': ['local']}


def preparar_pedido(pedido, pergunta=''):
    # Confere o pedido do modelo antes de executar. Devolve um dicionário com status:
    # 'ok' (pode executar), 'erro' (volta para o modelo) ou 'esclarecimento' (pergunta ao usuário).
    if not isinstance(pedido, dict) or pedido.get('name') not in CAMPOS or not isinstance(pedido.get('arguments'), dict):
        return dict(status='erro', mensagem='Erro: ferramenta ou arguments inválidos.')
    nome, argumentos = pedido['name'], pedido['arguments']
    if set(argumentos) - set(CAMPOS[nome]) - {'cenario'}:
        return dict(status='erro', mensagem='Erro: parâmetro extra. A velocidade é fixa em 15 km/h.')
    limpos = {}
    for campo in CAMPOS[nome]:
        texto = str(argumentos.get(campo) or '')
        local, opcoes = resolver_local(texto)
        if local is None:
            return dict(status='esclarecimento', campo=campo, mensagem=pergunta_ao_usuario(campo, opcoes))
        # O nome precisa ter vindo do usuário: evita que o modelo invente um local.
        if pergunta and not local_na_pergunta(local, pergunta):
            return dict(status='erro', mensagem=f'Erro: "{texto}" não aparece na pergunta. Copie o nome como o usuário escreveu.')
        limpos[campo] = local
    if nome == 'rota_tempo_previsto':
        cenario = resolver_cenario(argumentos.get('cenario') or 'tranquilo')
        if cenario is None:
            return dict(status='erro', mensagem='Erro: cenário deve ser tranquilo, movimentado ou cadeiras_no_patio.')
        if pergunta and cenario != 'tranquilo' and not aparece_na_pergunta(CENARIOS[cenario], pergunta):
            citados = [c for c in CENARIOS if c != 'tranquilo' and aparece_na_pergunta(CENARIOS[c], pergunta)]
            dica = 'Use cenario=' + citados[0] + '.' if citados else 'Omita cenario.'
            return dict(status='erro', mensagem='Erro: o usuário não pediu o cenário ' + cenario + '. ' + dica)
        limpos['cenario'] = cenario
    return dict(status='ok', pedido=dict(name=nome, arguments=limpos))


def executar(pedido):
    preparado = preparar_pedido(pedido)
    if preparado['status'] != 'ok':
        return preparado['mensagem']
    nome = preparado['pedido']['name']
    argumentos = preparado['pedido']['arguments']
    if nome == 'rota_tempo_previsto':
        return rota_tempo_previsto(**argumentos)
    return listar_vizinhos(**argumentos)


# Versões das regras (prompt). Guarde a versão junto dos resultados da avaliação.
# v1: Qwen explicava livremente. v2: base fixa de 15 km/h e uma só ferramenta de rota.
# v3: retorno direto da ferramenta, sem explicação. v4: volta ao ciclo do gabarito; explica copiando os números.
# v5: na v4 o Qwen respondeu 15 de 39 pedidos sem ferramenta, inventando rotas. As regras passam a dizer
#     que ele não conhece o mapa, e o código devolve a pergunta ao Qwen se ele responder rota sem ferramenta.
#     A lista dos 90 locais saiu das regras: o Qwen a usava para inventar nomes de corredor.
VERSAO_REGRAS = 'v5 (24/11/2026)'
REGRAS = (
    'Você é o assistente de rotas do carrinho no térreo do IESB. Você NÃO conhece o mapa, as distâncias '
    'nem os tempos: só as ferramentas sabem. '
    'Se a pergunta citar qualquer local, sala, mesa, banheiro ou lanchonete, chame uma ferramenta antes de responder, '
    'mesmo que a pergunta esteja informal, abreviada ou com erro de digitação. '
    'Para rota, caminho, distância ou tempo, use rota_tempo_previsto. '
    'Para saber o que está ligado diretamente a um local, use listar_vizinhos. '
    'Copie origem e destino como escritos pelo usuário, inclusive abreviações e erros: o código normalizará. '
    'Não troque origem e destino: "quero chegar a X saindo de Y" significa origem Y e destino X. '
    'Se faltar origem ou destino, envie texto vazio no campo. Se o nome for genérico, como "banheiro", '
    'envie o termo genérico, sem escolher bloco ou gênero. Nunca invente local nem posição atual. '
    'Exemplo: "biblio para b4" usa origem=biblio e destino=b4. '
    'A velocidade base é fixa em 15 km/h; não pergunte nem envie velocidade. '
    'Sem cenário, omita cenario (padrão tranquilo). Fluxo intenso ou lotado usa movimentado; '
    'cadeiras no pátio usa cadeiras_no_patio. Passar pelo pátio não é cenário de cadeiras. '
    'Uma chamada por vez, em <tool_call>{"name": "...", "arguments": {...}}</tool_call>. '
    'Se a ferramenta devolver erro, corrija a chamada. '
    'Só depois de receber o resultado da ferramenta, explique em português, em até quatro frases, '
    'copiando o caminho em uma linha e os números exatamente como vieram, sem converter unidades. '
    'Responda sem ferramenta somente a saudações e perguntas que não envolvem o mapa.'
)

PALAVRAS_DE_ROTA = ['rota', 'caminho', 'chegar', 'chego', 'ir', 'vou', 'pra', 'para', 'ate', 'distancia', 'tempo', 'vizinhos', 'ligados']


def fala_do_mapa(pergunta):
    # True se a pergunta cita um local, um nome genérico ou uma palavra de rota.
    texto = ' ' + padronizar(pergunta) + ' '
    nomes = list(NOMES) + list(GENERICOS) + PALAVRAS_DE_ROTA
    return any(' ' + nome + ' ' in texto for nome in nomes)


def extrair_json(texto):
    inicio = texto.find('{')
    fim = texto.rfind('}')
    if inicio == -1 or fim == -1:
        return None
    try:
        return json.loads(texto[inicio:fim + 1])
    except ValueError:
        return None


def ler_pedido_de_ferramenta(resposta):
    # Aceita também o JSON sem as marcas, que o Qwen 1.5B às vezes escreve: {"name": ..., "arguments": ...}
    if '<tool_call>' not in resposta and not resposta.strip().startswith('{'):
        return None              # o modelo não pediu ferramenta: é a resposta final
    return extrair_json(resposta)


def numeros_fora_da_ferramenta(resposta, resultados):
    # Números da explicação que não aparecem em nenhum retorno de ferramenta.
    # 22,3 = 22.3 e 46 = 46.0. Marcadores de lista ("1.", "2)") no início da linha são ignorados.
    def numeros(texto):
        texto = re.sub(r'(?m)^\s*\d+[.)]\s', ' ', texto)
        return set(float(n.replace(',', '.')) for n in re.findall(r'\d+(?:[.,]\d+)?', texto))
    fora = numeros(resposta) - numeros(' '.join(resultados))
    return sorted(f'{n:g}' for n in fora)


def agente(pergunta, mostrar=True, retornar_detalhes=False, historico=None):
    historico = historico or []
    mensagens = [{'role': 'system', 'content': REGRAS}] + historico + [{'role': 'user', 'content': pergunta}]
    # Texto escrito pelo usuário, usado para conferir se os nomes vieram do pedido.
    contexto = ' '.join([m['content'] for m in historico if m['role'] == 'user'] + [pergunta])
    detalhes = dict(resposta='', status='erro', chamadas=[], campo=None, tentativas=[], numeros_inventados=[])

    for passo in range(3):
        resposta = conversar(mensagens, FERRAMENTAS, max_tokens=300)    # 1. pensar
        detalhes['tentativas'].append(dict(resposta_modelo=resposta, geracao=ULTIMA_GERACAO.copy()))
        pedido = ler_pedido_de_ferramenta(resposta)

        if pedido is None and '<tool_call>' not in resposta:
            if not detalhes['chamadas'] and fala_do_mapa(contexto):
                # Respondeu sobre o mapa sem ferramenta: a resposta seria inventada. Pede de novo.
                mensagens.append({'role': 'assistant', 'content': resposta})
                mensagens.append({'role': 'user', 'content': 'Você não conhece o mapa. Responda somente com '
                                  '<tool_call>{"name": "rota_tempo_previsto", "arguments": {"origem": "...", "destino": "..."}}</tool_call>, '
                                  'copiando origem e destino desta pergunta (deixe vazio o que faltar). '
                                  'Se a pergunta for sobre o que está ao lado de um local, use listar_vizinhos. Pergunta: ' + pergunta})
                continue
            detalhes['resposta'] = resposta                            # 4. responder
            detalhes['status'] = 'concluido' if detalhes['chamadas'] else 'resposta_livre'
            break

        # O código confere o pedido antes de executar (nomes, cenário e parâmetros).
        if pedido is None:
            preparado = dict(status='erro', mensagem='Erro: JSON inválido. Envie exatamente uma chamada, com JSON completo, entre as marcas tool_call.')
        else:
            preparado = preparar_pedido(pedido, contexto)

        if preparado['status'] == 'esclarecimento':
            # Falta informação: o programa pergunta ao usuário em vez de escolher um local.
            detalhes.update(resposta=preparado['mensagem'], status='esclarecimento', campo=preparado['campo'])
            break

        if preparado['status'] == 'ok':
            resultado = executar(preparado['pedido'])                  # 2. agir
            detalhes['chamadas'].append(dict(pedido=preparado['pedido'], pedido_bruto=pedido,
                                            resultado=resultado))
        else:
            resultado = preparado['mensagem']                          # o erro volta para o modelo
        if mostrar:
            print('   [ferramenta]', preparado.get('pedido', pedido))
            print('   [resultado ]', resultado)

        # 3. observar: o resultado volta para o modelo, que vai explicar ao usuário.
        mensagens.append({'role': 'assistant', 'content': resposta})
        mensagens.append({'role': 'tool', 'content': resultado})
    else:
        detalhes['resposta'] = 'Não consegui consultar o mapa em 3 passos. Informe origem e destino com os nomes do mapa.'

    if detalhes['status'] == 'concluido':
        resultados = [c['resultado'] for c in detalhes['chamadas']]
        inventados = numeros_fora_da_ferramenta(detalhes['resposta'], resultados)
        detalhes['numeros_inventados'] = inventados
        if inventados:
            # Alucinação detectada: mostra também o retorno exato da ferramenta.
            detalhes['resposta'] += ('\n\n[Verificação] A explicação trouxe números que não vieram da ferramenta ('
                                     + ', '.join(inventados) + '). Resultado exato:\n' + resultados[-1])
    return detalhes if retornar_detalhes else detalhes['resposta']


## 8. Perguntas e descrição da ferramenta

In [ ]:
from transformers.utils import get_json_schema

print(json.dumps(get_json_schema(rota_tempo_previsto), indent=1, ensure_ascii=False))

perguntas = [
    "Como vou da Entrada Principal até a Mesa 4 e quanto tempo levo?",
    "Qual a rota, a distância e o tempo de B-04 até I-05?",
    "Da Mesa 1 até a Mesa 4, considere cadeiras no pátio.",
    "Quais locais estão ligados ao Pátio/Direito?",
]
for pergunta in perguntas:
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", agente(pergunta))
    print()


## 8.1 Interface opcional por texto
Execute `conversa_interativa()` depois de carregar o Qwen. Digite origem e destino como preferir. Se houver dúvida, o programa pede o local que falta: responda com o nome completo. As duas últimas trocas da conversa são enviadas de novo ao Qwen, para ele lembrar do pedido original. Use `cancelar` para começar outro pedido e `sair` para encerrar. A função não inicia automaticamente, para não bloquear a execução completa do notebook.


In [ ]:
def conversa_interativa():
    print("Conversa iniciada. Digite 'sair' para encerrar ou 'cancelar' para começar outra consulta.")
    historico = []
    while True:
        pergunta = input("Você: ").strip()
        if pergunta.lower() == 'sair':
            print('Conversa encerrada.')
            break
        if pergunta.lower() == 'cancelar':
            historico = []
            print('Consulta cancelada. Informe uma nova origem e destino.')
            continue
        if not pergunta:
            print('Digite sua pergunta ou o local solicitado.')
            continue
        resposta = agente(pergunta, historico=historico)
        print('Agente:', resposta)
        # Guarda só as duas últimas trocas, para o modelo lembrar de um esclarecimento.
        historico = (historico + [{'role': 'user', 'content': pergunta},
                                  {'role': 'assistant', 'content': resposta}])[-4:]


## 9. Avaliação — 39 pedidos de teste
Igual à Etapa 5 do gabarito, mas medindo o agente inteiro. Cada caso tem a pergunta, o que o agente deve fazer e os argumentos esperados:

- **ferramenta** (`rota_tempo_previsto` ou `listar_vizinhos`): deve chamar a ferramenta certa, com os nomes certos depois da padronização;
- **`None`**: saudação ou pergunta fora do mapa, deve responder sem ferramenta;
- **`'esclarecer'`**: falta informação ou o local não existe, deve perguntar ao usuário qual local, sem calcular rota.

Os casos incluem nomes exatos, vizinhos, os três cenários, escrita informal, abreviações (`b4`), erros de digitação (`bibliotca`), ordem invertida (“quero chegar em X saindo de Y”) e locais que não existem. A célula abaixo confere se todo caso de ferramenta roda sem erro antes de chamar o Qwen.


In [ ]:
# Cada caso: (pergunta, o que o agente deve fazer, argumentos esperados).
# O que fazer: nome da ferramenta, None (responder sem ferramenta) ou 'esclarecer' (perguntar ao usuário).
# Os argumentos são os nomes do mapa DEPOIS da padronização. Sem cenário = tranquilo.
R, V = 'rota_tempo_previsto', 'listar_vizinhos'
CASOS = [
    # Rotas com nomes exatos
    ('Como vou da Entrada Principal até a Mesa 2?', R, {'origem': 'Entrada Principal', 'destino': 'Mesa 2'}),
    ('Qual é a rota de B-04 para I-05?', R, {'origem': 'B-04', 'destino': 'I-05'}),
    ('Saindo da Biblioteca, como chego à C-02?', R, {'origem': 'Biblioteca', 'destino': 'C-02'}),
    ('Mostre o caminho de D-01 até WC Feminino Design.', R, {'origem': 'D-01', 'destino': 'WC Feminino Design'}),
    ('Como ir de J-06 para C-06?', R, {'origem': 'J-06', 'destino': 'C-06'}),
    ('Calcule a rota da Mesa 1 à Mesa 4.', R, {'origem': 'Mesa 1', 'destino': 'Mesa 4'}),
    ('Qual o caminho de WC Cinema até Anexo Creche?', R, {'origem': 'WC Cinema', 'destino': 'Anexo Creche'}),
    ('Quero ir do Bloco I até Café Mania. Qual a rota?', R, {'origem': 'Bloco I', 'destino': 'Café Mania'}),
    # Vizinhos
    ('Quais locais estão ligados diretamente ao Pátio/Direito?', V, {'local': 'Pátio/Direito'}),
    ('O que há ao lado do Corredor Design?', V, {'local': 'Corredor Design'}),
    ('Liste os vizinhos diretos de Cinema.', V, {'local': 'Cinema'}),
    ('Quais pontos têm ligação direta com Corredor I/J?', V, {'local': 'Corredor I/J'}),
    # Sem ferramenta
    ('Bom dia!', None, None),
    ('Obrigado pela ajuda!', None, None),
    ('Qual é a capital do Brasil?', None, None),
    # Cenários (Fase 2)
    ('Qual a rota e o tempo aprendido da Mesa 1 até a Mesa 4 no cenário tranquilo?', R, {'origem': 'Mesa 1', 'destino': 'Mesa 4'}),
    ('Qual a rota e o tempo previsto da Mesa 1 até a Mesa 4 no cenário movimentado?', R, {'origem': 'Mesa 1', 'destino': 'Mesa 4', 'cenario': 'movimentado'}),
    ('Qual o caminho da Mesa 1 à Mesa 4 no cenário sintético de cadeiras no pátio?', R, {'origem': 'Mesa 1', 'destino': 'Mesa 4', 'cenario': 'cadeiras_no_patio'}),
    ('Preveja a rota da Mesa 1 até Café Mania com atraso hipotético por cadeiras no pátio.', R, {'origem': 'Mesa 1', 'destino': 'Café Mania', 'cenario': 'cadeiras_no_patio'}),
    ('Qual o tempo aprendido de Easy Smart até Café Mania usando o cenário padrão?', R, {'origem': 'Easy Smart', 'destino': 'Café Mania'}),
    ('Calcule o tempo previsto saindo da Mesa 4 e chegando à própria Mesa 4 no cenário movimentado.', R, {'origem': 'Mesa 4', 'destino': 'Mesa 4', 'cenario': 'movimentado'}),
    ('No cenário sintético de cadeiras no pátio, como vou da Mesa 4 à Mesa 1 e qual o tempo previsto?', R, {'origem': 'Mesa 4', 'destino': 'Mesa 1', 'cenario': 'cadeiras_no_patio'}),
    ('Saindo do Café Mania, qual caminho devo seguir até a Biblioteca, qual a distância e o tempo previsto?', R, {'origem': 'Café Mania', 'destino': 'Biblioteca'}),
    # Escrita informal, abreviações, erros de digitação e ordem invertida
    ('to na biblio e quero ir pro cafe mania', R, {'origem': 'Biblioteca', 'destino': 'Café Mania'}),
    ('da b4 pra i5, qto tempo e qual caminho?', R, {'origem': 'B-04', 'destino': 'I-05'}),
    ('saindo da bibliotca ate mesa 2', R, {'origem': 'Biblioteca', 'destino': 'Mesa 2'}),
    ('cafe manai para easy smrt', R, {'origem': 'Café Mania', 'destino': 'Easy Smart'}),
    ('mesa um pra mesa quatro, ta lotado', R, {'origem': 'Mesa 1', 'destino': 'Mesa 4', 'cenario': 'movimentado'}),
    ('ENTRADA PRINCIPAL ate patio', R, {'origem': 'Entrada Principal', 'destino': 'Pátio/Direito'}),
    ('d1 pro banheiro fem do design', R, {'origem': 'D-01', 'destino': 'WC Feminino Design'}),
    ('quero chegar na biblioteca, saindo do cafe mania', R, {'origem': 'Café Mania', 'destino': 'Biblioteca'}),
    ('mesa 1 ate mesa 4 em cenario tranqüilo', R, {'origem': 'Mesa 1', 'destino': 'Mesa 4'}),
    ('biblio para biblio', R, {'origem': 'Biblioteca', 'destino': 'Biblioteca'}),
    # Falta informação: o agente deve perguntar, sem calcular rota
    ('da biblioteca para o banheiro', 'esclarecer', {'campo': 'destino'}),
    ('mesa 1 pra lanchonete', 'esclarecer', {'campo': 'destino'}),
    ('quero ir pra biblioteca', 'esclarecer', {'campo': 'origem'}),
    ('da biblioteca para sala 02', 'esclarecer', {'campo': 'destino'}),
    ('mesa 1 para auditorio', 'esclarecer', {'campo': 'destino'}),
    ('da biblioteca para B-07', 'esclarecer', {'campo': 'destino'}),
]

# Conferência dos próprios casos antes de chamar o Qwen: todo caso de ferramenta precisa executar sem erro.
for pergunta, esperada, argumentos in CASOS:
    if esperada in [R, V]:
        assert not executar({'name': esperada, 'arguments': argumentos}).startswith('Erro'), pergunta
print(len(CASOS), 'casos:', sum(1 for c in CASOS if c[1] == R), 'de rota,', sum(1 for c in CASOS if c[1] == V),
      'de vizinhos,', sum(1 for c in CASOS if c[1] is None), 'sem ferramenta e',
      sum(1 for c in CASOS if c[1] == 'esclarecer'), 'de esclarecimento.')


### 9.1 Rodar a avaliação com o Qwen
Para cada caso, mede: escolheu o que fazer certo? os argumentos estão certos? a explicação inventou algum número? quanto tempo levou? Um caso só conta como **certo** se tudo isso estiver certo. No fim, imprime o resumo, a lista de casos com erro (para a análise) e salva a tabela em `resultados_agente/`.

O tempo é por pedido completo (até 3 chamadas ao Qwen), sem contar o download do modelo.


In [ ]:
# Executar somente depois de carregar o Qwen real.
from statistics import median


def o_que_o_agente_fez(detalhes):
    if detalhes['status'] == 'esclarecimento':
        return 'esclarecer', {'campo': detalhes['campo']}
    if detalhes['chamadas']:
        pedido = detalhes['chamadas'][0]['pedido']
        return pedido['name'], pedido['arguments']
    return None, None


AVALIACAO = []
for pergunta, esperada, argumentos in CASOS:
    inicio = perf_counter()
    detalhes = agente(pergunta, mostrar=False, retornar_detalhes=True)
    segundos = perf_counter() - inicio

    escolhida, recebidos = o_que_o_agente_fez(detalhes)
    if esperada == R:
        argumentos = dict({'cenario': 'tranquilo'}, **argumentos)   # sem cenário = tranquilo
    ferramenta_ok = escolhida == esperada
    argumentos_ok = ferramenta_ok and recebidos == argumentos
    inventou = detalhes['numeros_inventados']

    if ferramenta_ok and argumentos_ok and not inventou:
        marca = 'certo '
    else:
        marca = 'ERRADO'
    print(marca, '|', pergunta, '| esperada:', esperada, '| escolhida:', escolhida,
          '| inventados:', inventou, f'| {segundos:.1f} s')
    AVALIACAO.append(dict(pergunta=pergunta, esperada=esperada, escolhida=escolhida,
                           argumentos_esperados=argumentos, argumentos_recebidos=recebidos,
                           ferramenta_ok=ferramenta_ok, argumentos_ok=argumentos_ok,
                           numeros_inventados=inventou, segundos=segundos,
                           status=detalhes['status'], resposta=detalhes['resposta']))

tabela = pd.DataFrame(AVALIACAO)
com_rota = tabela[tabela.esperada.isin([R, V])]
print()
print('Modelo:', nome_qwen, '| dispositivo:', dispositivo, '| regras:', VERSAO_REGRAS)
print('Escolheu o que fazer certo em', tabela.ferramenta_ok.sum(), 'de', len(tabela), 'casos.')
print('Argumentos certos em', tabela.argumentos_ok.sum(), 'de', len(tabela), 'casos.')
print('Rota ou vizinhos válidos (ferramenta e argumentos certos) em', com_rota.argumentos_ok.sum(), 'de', len(com_rota), 'casos.')
print('Explicações com número inventado:', (com_rota.numeros_inventados.str.len() > 0).sum(), 'de', len(com_rota))
print(f'Tempo por pedido: mediana {tabela.segundos.median():.1f} s, máximo {tabela.segundos.max():.1f} s.')

print('\nCasos com erro, para a análise:')
erros = tabela[~tabela.argumentos_ok | (tabela.numeros_inventados.str.len() > 0)]
for _, linha in erros.iterrows():
    print('-', linha.pergunta)
    print('  esperado:', linha.esperada, linha.argumentos_esperados, '| recebido:', linha.escolhida, linha.argumentos_recebidos)
    print('  resposta:', linha.resposta[:200])

Path('resultados_agente').mkdir(exist_ok=True)
tabela.to_csv('resultados_agente/avaliacao_' + VERSAO_REGRAS.split()[0] + '.csv', index=False)


### Sua análise

Execuções de 24/11/2026 com as regras v5 e o Qwen2.5-1.5B-Instruct: uma no Colab, sem GPU, e outra numa GPU RTX 4050 local. As duas deram o mesmo resultado em todos os 39 casos, porque a geração é determinística (`do_sample=False`).

**1. A escolha.** O agente fez o que devia em **31 de 39 casos** (rota, vizinhos, resposta sem ferramenta ou esclarecimento).

| Grupo de casos | Certos |
|---|---|
| Nomes exatos | 7 de 8 |
| Vizinhos | 3 de 4 |
| Sem ferramenta (saudação, fora do mapa) | 3 de 3 |
| Cenários da Fase 2 | 7 de 8 |
| Escrita informal, abreviação, erro de digitação | 3 de 10 |
| Esclarecimento (local genérico ou inexistente) | 4 de 6 |

O ponto fraco é a escrita informal. Em “cafe manai para easy smrt”, “mesa um pra mesa quatro, ta lotado” e “biblio para biblio”, o Qwen não chamou a ferramenta nem depois de o código pedir de novo. Ele escreveu coisas como “Easy Smart para Café Manai, caminha 203” e “origem=ma1 destinaçao=ma4”, ou repetiu a pergunta (“da biblioteca para B-07”). Em “O que há ao lado do Corredor Design?”, primeiro inventou “Sala 203, Sala 204…” e, quando o código pediu a ferramenta, chamou a rota sem destino, em vez de `listar_vizinhos`. Quando o Qwen chama a ferramenta, a padronização do código resolve bem os nomes informais: `b4` → B-04, `biblio` → Biblioteca, “banheiro” → pergunta qual.

**2. Os argumentos.** Nos 31 casos em que o agente chamou uma ferramenta ou pediu esclarecimento, os argumentos estavam certos depois da padronização. Ele não trocou origem e destino, nem em “quero chegar na biblioteca, saindo do cafe mania”. Durante o desenvolvimento, o Qwen tentou usar o cenário `movimentado` num pedido sobre cadeiras no pátio; a conferência do código recusou e indicou o cenário certo.

**3. A explicação.** Em **4 de 30** explicações apareceu um número que não estava no resultado da ferramenta: 74,2 s virou “74 segundos”, 51,8 s virou “51 segundos”, 22,2 s virou “22 segundos” e 17,8 s virou “cerca de 17 segundos”. Todos foram sinalizados por `numeros_fora_da_ferramenta`, e o agente mostrou o resultado exato. O Qwen também reescreve o caminho (“Caminhar pela área interna do prédio…”) e às vezes acrescenta conselhos que não vieram da ferramenta, como “use o transporte público”. Esses conselhos a verificação por números não detecta. Na v4, sem a proteção da v5, 15 pedidos receberam rotas totalmente inventadas (“78 metros”, “Corredor Bloco C - 04/07”); na v5, nenhuma rota inventada chegou ao usuário.

**4. Custo.** O modelo roda dentro da sessão: não há API paga nem chave no código, mas há download (cerca de 3 GB) e tempo de processamento. Com GPU (RTX 4050), a mediana foi de **3,8 s por pedido**, com máximo de 9,5 s. No Colab sem GPU, foi de **135 s por pedido**, com máximo de 234 s, e a avaliação inteira levou 88 minutos, cerca de 35 vezes mais. Pedidos com erro custam mais, porque o agente chama o Qwen até 3 vezes.

**5. O seminário.** O caso de falha para mostrar ao vivo é **“cafe manai para easy smrt”**. O Qwen 1.5B não entende o pedido com erro de digitação e não chama a ferramenta; nas três tentativas escreve “Easy Smart para Café Manai, caminha 203”. O agente responde “Não consegui consultar o mapa em 3 passos”, em vez de mostrar uma rota inventada. Isso mostra o limite do modelo pequeno e a proteção do código. A mesma pergunta com os nomes certos (“Café Mania para Easy Smart”) funciona.


## Limites e próximos passos
Este notebook reúne as três fases. A seção 5.3 treina dois modelos com dados sintéticos e integra o escolhido ao A*. A base é fixa em 15 km/h e o Qwen tem uma única ferramenta de rota; a fórmula distância/velocidade fica apenas como referência didática do Marco 1. Desde a v4, o agente segue o ciclo do gabarito: o resultado da ferramenta volta ao Qwen, que explica a rota, e números fora do resultado são sinalizados. A v5 impede respostas sobre o mapa sem ferramenta. As execuções v1 a v3 estão no histórico Git. A v5 foi executada do início ao fim em 24/11/2026 numa GPU RTX 4050 (não no Colab): 31 de 39 casos certos, mediana de 3,8 s por pedido. Ainda é preciso repetir a execução no Colab com T4 antes da apresentação.

O grafo agora divide os corredores em pontos sucessivos, agrupando portas próximas. O pátio tem trajetos alternativos condicionados à disposição de mesas e cadeiras; o programa não detecta bloqueios físicos. As posições e distâncias continuam aproximadas. As ligações e as distâncias precisam ser revisadas com medições para representar o campus com maior fidelidade.

O Qwen é executado localmente na sessão e não faz chamada a uma API paga nem usa chave escrita no código. Isso não significa custo zero: há tempo de inferência, consumo de memória e uso de CPU/GPU, que dependem do ambiente do Colab. A avaliação registra a duração por caso. Depois da execução real, o relatório deve discutir respostas inventadas, chamadas incorretas, pedidos fora do mapa e o efeito desses custos computacionais.